In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

SILVER_PATH = "output/silver"
GOLD_PATH = "output/gold"

In [2]:
orders = spark.read.csv(f"{SILVER_PATH}/orders", header=True, inferSchema=True)
order_items = spark.read.csv(f"{SILVER_PATH}/order_items", header=True, inferSchema=True)
customers = spark.read.csv(f"{SILVER_PATH}/customers", header=True, inferSchema=True)
products = spark.read.csv(f"{SILVER_PATH}/products", header=True, inferSchema=True)
sellers = spark.read.csv(f"{SILVER_PATH}/sellers", header=True, inferSchema=True)
payments = spark.read.csv(f"{SILVER_PATH}/payments", header=True, inferSchema=True)

In [3]:
orders = orders \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))

order_items = order_items \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

payments = payments \
    .withColumn("payment_value", col("payment_value").cast("double"))

In [4]:
customer_orders = customers.alias("c") \
    .join(orders.alias("o"), "customer_id") \
    .join(order_items.alias("oi"), "order_id", "left") \
    .select(
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
        "order_id",
        "order_purchase_timestamp",
        (col("price") + col("freight_value")).alias("order_value")
    )

window_latest = Window.partitionBy("customer_unique_id").orderBy(col("order_purchase_timestamp").desc())

latest_customer = customer_orders \
    .withColumn("rn", row_number().over(window_latest)) \
    .filter("rn = 1") \
    .drop("rn")

customer_agg = customer_orders.groupBy("customer_unique_id") \
    .agg(
        min("order_purchase_timestamp").alias("first_order_date"),
        countDistinct("order_id").alias("total_orders"),
        sum("order_value").alias("total_spend")
    )

dim_customers = latest_customer \
    .join(customer_agg, "customer_unique_id") \
    .withColumn("customer_key", sha2(col("customer_unique_id"), 256)) \
    .withColumn("is_repeat_customer", col("total_orders") > 1) \
    .select(
        "customer_key",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
        "first_order_date",
        "total_orders",
        "total_spend",
        "is_repeat_customer"
    )

In [5]:
dim_products = products \
    .withColumn("product_key", sha2(col("product_id"), 256)) \
    .withColumn("product_category_name",
        coalesce(col("product_category_name"), lit("unknown"))
    ) \
    .withColumn("product_volume_cm3",
        when(
            col("product_length_cm").isNotNull() &
            col("product_height_cm").isNotNull() &
            col("product_width_cm").isNotNull(),
            col("product_length_cm") *
            col("product_height_cm") *
            col("product_width_cm")
        )
    ) \
    .select(
        "product_key",
        "product_id",
        "product_category_name",
        col("product_weight_g").cast("double"),
        "product_volume_cm3",
        "product_photos_qty",
        "product_description_lenght"
    )

In [6]:
dim_sellers = sellers \
    .withColumn("seller_key", sha2(col("seller_id"), 256)) \
    .select(
        "seller_key",
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix"
    )

In [7]:
window_payment = Window.partitionBy("order_id") \
    .orderBy(col("payment_value").desc(), col("payment_type"))

payment_ranked = payments.withColumn("rn", row_number().over(window_payment))

payment_main = payment_ranked.filter("rn = 1") \
    .select("order_id", "payment_type")

payment_summary = payments.groupBy("order_id") \
    .agg(
        sum("payment_value").alias("total_payment"),
        max("payment_installments").alias("payment_installments")
    )

payment_agg = payment_summary.join(payment_main, "order_id")

In [8]:
order_item_enriched = order_items \
    .join(orders, "order_id") \
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )

In [9]:
order_totals = order_items.groupBy("order_id") \
    .agg(sum("price").alias("total_price"))

fact_base = order_item_enriched \
    .join(order_totals, "order_id") \
    .join(payment_agg, "order_id", "left") \
    .withColumn(
        "payment_value",
        (col("price") / col("total_price")) * col("total_payment")
    )

In [10]:
fact_order_items = fact_base \
    .join(customers.select("customer_id", "customer_unique_id"), "customer_id") \
    .withColumn("customer_key", sha2(col("customer_unique_id"), 256)) \
    .withColumn("product_key", sha2(col("product_id"), 256)) \
    .withColumn("seller_key", sha2(col("seller_id"), 256)) \
    .withColumn("order_item_sk",
        sha2(concat_ws("-", col("order_id"), col("order_item_id")), 256)
    ) \
    .withColumn("order_date", to_date("order_purchase_timestamp")) \
    .withColumn("days_to_deliver",
        when(col("order_delivered_customer_date").isNotNull(),
             datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    ) \
    .withColumn("days_delivery_vs_estimate",
        when(col("order_delivered_customer_date").isNotNull(),
             datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
    ) \
    .withColumn("is_late_delivery",
        when(col("days_delivery_vs_estimate").isNotNull(),
             col("days_delivery_vs_estimate") > 0)
    ) \
    .select(
        "order_item_sk",
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "seller_key",
        "order_date",
        "order_status",
        col("price").cast("double"),
        col("freight_value").cast("double"),
        "payment_value",
        "payment_type",
        "payment_installments",
        "days_to_deliver",
        "days_delivery_vs_estimate",
        "is_late_delivery"
    )

In [11]:
dim_customers.write.mode("overwrite").csv(f"{GOLD_PATH}/dim_customers", header=True)
dim_products.write.mode("overwrite").csv(f"{GOLD_PATH}/dim_products", header=True)
dim_sellers.write.mode("overwrite").csv(f"{GOLD_PATH}/dim_sellers", header=True)
fact_order_items.write.mode("overwrite").csv(f"{GOLD_PATH}/fact_order_items", header=True)

In [12]:
print("fact_order_items:", fact_order_items.count())
print("dim_customers:", dim_customers.count())
print("dim_products:", dim_products.count())
print("dim_sellers:", dim_sellers.count())

fact_order_items: 14480
dim_customers: 12766
dim_products: 32951
dim_sellers: 3095


In [13]:
fact_alias = fact_order_items.alias("f")
cust_alias = dim_customers.alias("c")

invalid_customers = fact_alias \
    .join(cust_alias, col("f.customer_key") == col("c.customer_key"), "left") \
    .filter(col("c.customer_key").isNull()) \
    .count()

print("Invalid customer_key:", invalid_customers)

Invalid customer_key: 0


In [14]:
prod_alias = dim_products.alias("p")

invalid_products = fact_alias \
    .join(prod_alias, col("f.product_key") == col("p.product_key"), "left") \
    .filter(col("p.product_key").isNull()) \
    .count()

print("Invalid product_key:", invalid_products)

Invalid product_key: 0


In [15]:
sell_alias = dim_sellers.alias("s")

invalid_sellers = fact_alias \
    .join(sell_alias, col("f.seller_key") == col("s.seller_key"), "left") \
    .filter(col("s.seller_key").isNull()) \
    .count()

print("Invalid seller_key:", invalid_sellers)

Invalid seller_key: 0


In [16]:
assert invalid_customers == 0, "Customer FK violation!"
assert invalid_products == 0, "Product FK violation!"
assert invalid_sellers == 0, "Seller FK violation!"